In [1]:
# ==========================================
# Notebook 16
# End-to-End Personalized Recommendation System
# ==========================================

import pandas as pd
import numpy as np

import faiss

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

from transformers import pipeline

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

interactions_df = pd.read_csv("../data/user_interactions.csv")

hybrid_df = pd.read_csv("../data/hybrid_recommendations.csv")

evaluation_df = pd.read_csv("../data/recommendation_evaluation.csv")

In [3]:
item_embeddings = np.load("../data/item_embeddings.npy")

user_embeddings = np.load("../data/user_embeddings.npy")

In [4]:
index = faiss.read_index("../data/item_index.faiss")

In [5]:
print("Items:", len(profiles_df))

print("Users:", interactions_df["user_id"].nunique())

print("Recommendations:", len(hybrid_df))

Items: 10
Users: 5
Recommendations: 25


In [6]:
llm = pipeline(
    "text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=300
)

Device set to use cpu


In [7]:
def get_user_history(user_id):

    history = interactions_df[interactions_df["user_id"] == user_id]

    history = pd.merge(
        history, profiles_df[["item_id", "title", "category"]], on="item_id", how="left"
    )

    return history

In [8]:
def build_user_profile(user_id):

    history = interactions_df[interactions_df["user_id"] == user_id]

    item_vectors = []

    for item_id in history["item_id"]:

        idx = profiles_df[profiles_df["item_id"] == item_id].index[0]

        item_vectors.append(item_embeddings[idx])

    return np.mean(item_vectors, axis=0)

In [9]:
def retrieve_candidates(user_vector, top_k=20):

    distances, indices = index.search(
        user_vector.reshape(1, -1).astype("float32"), top_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        results.append(
            {
                "rank": rank + 1,
                "item_id": profiles_df.iloc[idx]["item_id"],
                "title": profiles_df.iloc[idx]["title"],
                "category": profiles_df.iloc[idx]["category"],
            }
        )

    return pd.DataFrame(results)

In [10]:
def apply_mmr(user_vector, candidates, top_k=5, lambda_param=0.7):

    candidate_embeddings = []

    for item_id in candidates["item_id"]:

        idx = profiles_df[profiles_df["item_id"] == item_id].index[0]

        candidate_embeddings.append(item_embeddings[idx])

    candidate_embeddings = np.array(candidate_embeddings)

    selected = []

    candidate_indices = list(range(len(candidates)))

    relevance_scores = cosine_similarity(
        user_vector.reshape(1, -1), candidate_embeddings
    )[0]

    while len(selected) < min(top_k, len(candidate_indices)):

        mmr_scores = []

        for idx in candidate_indices:

            relevance = relevance_scores[idx]

            if len(selected) == 0:

                diversity = 0

            else:

                diversity = max(
                    [
                        cosine_similarity(
                            candidate_embeddings[idx].reshape(1, -1),
                            candidate_embeddings[s].reshape(1, -1),
                        )[0][0]
                        for s in selected
                    ]
                )

            score = lambda_param * relevance - (1 - lambda_param) * diversity

            mmr_scores.append((idx, score))

        best = max(mmr_scores, key=lambda x: x[1])[0]

        selected.append(best)

        candidate_indices.remove(best)

    return candidates.iloc[selected]

In [11]:
def llm_rerank(user_id, recommendations):

    history = get_user_history(user_id)

    history_titles = history["title"].tolist()

    candidate_text = "\n".join(recommendations["title"].tolist())

    prompt = f"""
    User History:

    {history_titles}

    Candidate Recommendations:

    {candidate_text}

    Select the best 3 recommendations.

    Explain why.

    Return concise output.
    """

    response = llm(prompt)

    return response[0]["generated_text"]

In [12]:
def generate_recommendations(user_id):

    user_vector = build_user_profile(user_id)

    candidates = retrieve_candidates(user_vector, top_k=20)

    diversified = apply_mmr(user_vector, candidates, top_k=10)

    explanation = llm_rerank(user_id, diversified)

    return {
        "user_id": user_id,
        "recommendations": diversified,
        "llm_explanation": explanation,
    }

In [13]:
result = generate_recommendations(101)

In [14]:
result["recommendations"]

,rank,item_id,title,category
0,1,10,Large Language Models,Artificial Intelligence
1,2,10,Large Language Models,Artificial Intelligence
2,3,10,Large Language Models,Artificial Intelligence
3,4,10,Large Language Models,Artificial Intelligence
4,5,10,Large Language Models,Artificial Intelligence
5,6,10,Large Language Models,Artificial Intelligence
6,7,10,Large Language Models,Artificial Intelligence
7,8,10,Large Language Models,Artificial Intelligence
8,9,10,Large Language Models,Artificial Intelligence
9,10,10,Large Language Models,Artificial Intelligence


In [15]:
print(result["llm_explanation"])


    User History:

    ['The Early Days of Stripe', 'Building SpaceX', 'Cloud Computing Fundamentals']

    Candidate Recommendations:

    Large Language Models
Large Language Models
Large Language Models
Large Language Models
Large Language Models
Large Language Models
Large Language Models
Large Language Models
Large Language Models
Large Language Models

    Select the best 3 recommendations.

    Explain why.

    Return concise output.
     "Large Language Models" is a popular choice for generating text, especially in fields like marketing and product development. It can be used to generate coherent sentences about various topics such as startups, software engineering, and technology trends. By selecting large language models, you can ensure that your response is well-structured and engaging.

### Output:
```python
["Large Language Models"]
```

This recommendation ensures that your responses are informative and structured, which aligns with the requirement of providing comprehe

In [16]:
all_results = []

In [17]:
for user_id in sorted(interactions_df["user_id"].unique()):

    output = generate_recommendations(user_id)

    all_results.append(
        {
            "user_id": user_id,
            "recommendations": output["recommendations"]["title"].tolist(),
            "llm_explanation": output["llm_explanation"],
        }
    )

In [18]:
results_df = pd.DataFrame(all_results)

results_df.head()

,user_id,recommendations,llm_explanation
0,101,"[Large Language Models, Large Language Models,...",\n User History:\n\n ['The Early Days of...
1,102,"[Large Language Models, Large Language Models,...",\n User History:\n\n ['The Early Days of...
2,103,"[Large Language Models, Large Language Models,...",\n User History:\n\n ['The Psychology of...
3,104,"[Large Language Models, Large Language Models,...",\n User History:\n\n ['AI for Healthcare...
4,105,"[Large Language Models, Large Language Models,...",\n User History:\n\n ['Cloud Computing F...


In [19]:
results_df.to_csv("../data/final_recommendations.csv", index=False)

In [20]:
saved_df = pd.read_csv("../data/final_recommendations.csv")

saved_df.head()

,user_id,recommendations,llm_explanation
0,101,"['Large Language Models', 'Large Language Mode...",\n User History:\n\n ['The Early Days of...
1,102,"['Large Language Models', 'Large Language Mode...",\n User History:\n\n ['The Early Days of...
2,103,"['Large Language Models', 'Large Language Mode...",\n User History:\n\n ['The Psychology of...
3,104,"['Large Language Models', 'Large Language Mode...",\n User History:\n\n ['AI for Healthcare...
4,105,"['Large Language Models', 'Large Language Mode...",\n User History:\n\n ['Cloud Computing F...


In [21]:
evaluation_df

,user_id,precision_at_5,recall_at_5,ndcg_at_5,diversity,novelty
0,101,0.0,0.0,0,0.513415,0.9
1,102,0.0,0.0,0,0.543107,0.8
2,103,0.0,0.0,0,0.503253,0.8
3,104,0.0,0.0,0,0.557056,0.8
4,105,0.0,0.0,0,0.505819,0.8


In [22]:
evaluation_df.mean(numeric_only=True)

user_id           103.00000
precision_at_5      0.00000
recall_at_5         0.00000
ndcg_at_5           0.00000
diversity           0.52453
novelty             0.82000
dtype: float64